# Nifty 500 Dataset — Comprehensive Audit V2

This notebook performs a reproducible, read-only audit of the extracted dataset. It is designed to work with the `DE/` directory from the supplied archives and to tolerate different dataset versions.

**Important:** an automated audit can establish structural and numerical consistency. It cannot independently prove that vendor data is historically true, that a corporate-action adjustment is correct, or that fundamentals were public on a particular date without authoritative source records. Such cases are marked `UNVERIFIED` or `BLOCKED`, not silently passed.

## What this notebook checks

1. File inventory, metadata sidecars, Parquet readability, schemas and hashes.
2. Price files: keys, dates, OHLC relationships, volume, missingness, duplicates, gaps and suspicious returns.
3. Existing security mapping/security master: schema, duplicate mappings, unmatched identifiers, coverage and ambiguity.
4. Historical membership: snapshots or effective intervals, duplicate keys, constituent counts and date coverage.
5. Fundamentals: keys, duplicate records, missingness, numeric anomalies and point-in-time availability fields.
6. Corporate-action evidence and price-adjustment uncertainty.
7. Existing feature data: schema, duplicate keys, missing/infinite values and independent recomputation of common indicators where possible.
8. Cross-dataset coverage and strategy-specific readiness gates.
9. Machine-readable CSV/JSON reports. No source files are overwritten.

Run from top to bottom. The audit can take several minutes because it reads the price Parquet files.

## 0. Environment and configuration

Run the install cell once if required. The configured default matches the project path previously used. If your extracted archive is elsewhere, change `DATA_ROOT`.

In [1]:
%pip install -q pandas numpy pyarrow fastparquet

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import os, re, json, hashlib, warnings, math, sys, platform
from datetime import datetime, timezone
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore", category=FutureWarning)

# CHANGE THIS if your extracted DE folder is somewhere else.
DATA_ROOT = Path.home() / "Documents/vscode/BacktestEngine/dataextracted"

# Reports are written OUTSIDE the source dataset.
REPORT_ROOT = DATA_ROOT.parent / "audit_v2_output"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

# Audit thresholds. These flag observations for review; they do not automatically delete data.
MAX_GAP_CALENDAR_DAYS = 7
EXTREME_ABS_DAILY_RETURN = 0.50       # investigate daily close-to-close moves > 50%
MAX_OHLC_REL_TOLERANCE = 1e-6
FEATURE_ABS_TOLERANCE = 1e-6
FEATURE_REL_TOLERANCE = 1e-4
SAMPLE_ROWS_FOR_SCHEMA = 5

assert DATA_ROOT.exists(), f"DATA_ROOT does not exist: {DATA_ROOT}"
print("Data root:", DATA_ROOT.resolve())
print("Report root:", REPORT_ROOT.resolve())
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| numpy:", np.__version__)

Data root: /home/rishi/Documents/vscode/BacktestEngine/dataextracted
Report root: /home/rishi/Documents/vscode/BacktestEngine/audit_v2_output
Python: 3.14.9 | pandas: 3.1.1 | numpy: 2.5.4


## 1. Common utilities and issue register

The notebook distinguishes `FAIL` (objective rule violated), `REVIEW` (requires investigation), `UNVERIFIED` (the supplied data cannot prove the claim), and `PASS` (the specific test passed). A `PASS` is never broader than the test performed.

In [2]:
issues = []
metrics = {}
file_inventory_rows = []
parquet_schema_rows = []
read_errors = []
price_file_rows = []
price_frames = []
dataset_frames = {}

def add_issue(severity, area, code, message, file_path=None, security=None, date=None, evidence=None):
    issues.append({
        "severity": severity, "area": area, "code": code, "message": str(message),
        "file_path": str(file_path) if file_path is not None else None,
        "security": security, "date": str(date) if date is not None else None,
        "evidence": str(evidence) if evidence is not None else None
    })

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def is_metadata_path(p):
    parts = [x.lower() for x in p.parts]
    return "__macosx" in parts or p.name.startswith("._") or p.name in {".ds_store"}

def discover_files():
    return [p for p in DATA_ROOT.rglob("*") if p.is_file() and not is_metadata_path(p)]

def find_files(tokens=(), suffixes=None):
    out = []
    for p in discover_files():
        if suffixes and p.suffix.lower() not in suffixes:
            continue
        low = str(p.relative_to(DATA_ROOT)).lower()
        if all(t.lower() in low for t in tokens):
            out.append(p)
    return sorted(out)

def read_table(path):
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in {".csv", ".txt"}:
        # utf-8-sig handles common BOMs; fallback is explicit and recorded by caller.
        return pd.read_csv(path, low_memory=False, encoding="utf-8-sig")
    raise ValueError(f"Unsupported table type: {path}")

def norm_col(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).strip().lower())

def pick_col(columns, candidates):
    lookup = {norm_col(c): c for c in columns}
    for c in candidates:
        if norm_col(c) in lookup:
            return lookup[norm_col(c)]
    # Conservative fuzzy fallback: exact token-like containment only.
    for c in columns:
        n = norm_col(c)
        for cand in candidates:
            nc = norm_col(cand)
            if nc and (n == nc or n.endswith(nc) or n.startswith(nc)):
                return c
    return None

def to_datetime_series(s):
    # Do not silently coerce unparseable values without reporting them.
    return pd.to_datetime(s, errors="coerce", utc=False)

def numeric_series(s):
    return pd.to_numeric(s, errors="coerce")

def write_report(df, name):
    path = REPORT_ROOT / name
    df.to_csv(path, index=False)
    return path

def summarize_missing(df, dataset, key_cols=()):
    rows = []
    n = len(df)
    for c in df.columns:
        miss = int(df[c].isna().sum())
        if miss:
            rows.append({"dataset": dataset, "column": c, "rows": n, "missing": miss,
                         "missing_pct": round(100*miss/n, 4) if n else np.nan})
    return rows

print("Utilities ready.")

Utilities ready.


## 2. File inventory, Parquet integrity and schema discovery

This excludes macOS `__MACOSX` and `._*` sidecars before inspecting actual data. Every real Parquet file is checked for readable metadata and a full table read. The full read catches corruption that metadata-only inspection can miss.

In [3]:
all_files = discover_files()
for p in all_files:
    rel = str(p.relative_to(DATA_ROOT))
    row = {"path": rel, "suffix": p.suffix.lower(), "bytes": p.stat().st_size,
           "sha256": sha256_file(p) if p.stat().st_size < 2_000_000_000 else None}
    file_inventory_rows.append(row)

    if p.suffix.lower() == ".parquet":
        try:
            pf = pq.ParquetFile(p)
            schema = pf.schema_arrow
            df = pd.read_parquet(p)
            parquet_schema_rows.append({
                "path": rel, "rows_metadata": pf.metadata.num_rows,
                "rows_read": len(df), "columns": len(df.columns),
                "column_names": "|".join(map(str, df.columns)),
                "schema": str(schema), "readable": True,
                "row_count_matches_metadata": pf.metadata.num_rows == len(df)
            })
        except Exception as e:
            read_errors.append({"path": rel, "error": repr(e)})
            add_issue("FAIL", "inventory", "PARQUET_READ_FAILURE", repr(e), p)

inventory_df = pd.DataFrame(file_inventory_rows)
schema_df = pd.DataFrame(parquet_schema_rows)
read_errors_df = pd.DataFrame(read_errors)

print("Actual files:", len(all_files))
print("Parquet:", sum(p.suffix.lower()==".parquet" for p in all_files))
print("CSV:", sum(p.suffix.lower()==".csv" for p in all_files))
print("Metadata sidecars excluded:", sum(1 for p in DATA_ROOT.rglob("*") if p.is_file() and is_metadata_path(p)))
print("Unreadable Parquet:", len(read_errors))
display(inventory_df.groupby("suffix").agg(files=("path","count"), total_MB=("bytes",lambda x: round(x.sum()/1e6,2))))
display(read_errors_df.head(30))

Actual files: 931
Parquet: 877
CSV: 36
Metadata sidecars excluded: 960
Unreadable Parquet: 0


,files,total_MB
suffix,,
,1,0.01
.csv,36,5.03
.ipynb,15,16.61
.json,2,0.44
.parquet,877,451.28


""


## 3. Identify datasets and inspect schemas

This section detects likely files by path and filename. It does not assume a single archive version. In particular, it looks for the existing `historical_membership_price_mapping_audit.csv` and/or `security_master.parquet`; it does not regenerate or overwrite either.

In [4]:
def show_candidates(label, paths):
    print(f"\\n{label}: {len(paths)} candidate(s)")
    for p in paths[:30]:
        try:
            d = read_table(p)
            print(f"- {p.relative_to(DATA_ROOT)} | shape={d.shape} | columns={list(d.columns)}")
        except Exception as e:
            print(f"- {p.relative_to(DATA_ROOT)} | READ ERROR: {e}")

membership_files = [p for p in discover_files()
                    if "membership" in p.name.lower()
                    and "mapping" not in p.name.lower()
                    and "audit" not in p.name.lower()
                    and p.suffix.lower() in {".parquet",".csv"}]
security_master_files = [p for p in discover_files() if "security_master" in p.name.lower() and p.suffix.lower() in {".parquet",".csv"}]
mapping_audit_files = [p for p in discover_files() if "mapping" in p.name.lower() and "audit" in p.name.lower() and p.suffix.lower() in {".parquet",".csv"}]
fundamental_files = [p for p in discover_files() if "fundamental" in p.name.lower() and "failed" not in p.name.lower() and p.suffix.lower() in {".parquet",".csv"}]
feature_files = [p for p in discover_files() if ("feature" in p.name.lower()) and p.suffix.lower() in {".parquet",".csv"}]
corporate_action_files = [p for p in discover_files() if any(x in p.name.lower() for x in ["corporate_action","corporateaction","split","bonus","dividend","rights_issue","demerger"]) and p.suffix.lower() in {".parquet",".csv"}]

show_candidates("Membership", membership_files)
show_candidates("Security master", security_master_files)
show_candidates("Mapping audit", mapping_audit_files)
show_candidates("Fundamentals", fundamental_files)
show_candidates("Feature datasets", feature_files)
show_candidates("Corporate-action candidates", corporate_action_files)

if not corporate_action_files:
    add_issue("UNVERIFIED", "corporate_actions", "NO_CORPORATE_ACTION_SOURCE_FOUND",
              "No actual corporate-action data file was discovered. This does not prove no data exists outside the scanned archive.")

\nMembership: 2 candidate(s)
- DE/data/raw/constituents/nifty500_membership_validation.csv | shape=(20, 4) | columns=['Date', 'Unique_Tickers', 'Expected_500', 'Difference_From_500']
- DE/data/raw/constituents/nifty500_historical_membership.parquet | shape=(9857, 2) | columns=['Date', 'Ticker']
\nSecurity master: 1 candidate(s)
- DE/data/raw/constituents/security_master.parquet | shape=(500, 4) | columns=['Bloomberg_Index_Code', 'Bloomberg_ID', 'Index', 'Snapshot_Date']
\nMapping audit: 1 candidate(s)
- DE/data/raw/constituents/historical_membership_price_mapping_audit.csv | shape=(9857, 5) | columns=['Date', 'Ticker', 'Membership_Root', 'Price_Match_Count', 'Mapping_Status']
\nFundamentals: 24 candidate(s)
- DE/data/raw/fundamentals/nifty500_historical_fundamental_incremental_field_exceptions.parquet | shape=(0, 0) | columns=[]
- DE/data/raw/fundamentals/potentially_invalid_fundamental_values.csv | shape=(511, 4) | columns=['Bloomberg_Security', 'Date', 'Value', 'Field']
- DE/data/raw

## 4. Price data audit

Price files are read from folders named `prices` or `prices_daily`, excluding extraction logs. The code standardizes common column aliases but reports missing required fields. Ticker identity is taken from an in-file ticker/security column when present; otherwise the filename stem is used as a provisional file identifier.

**Calendar limitation:** weekends and long calendar gaps are not equivalent to missing exchange sessions. Without an authoritative NSE trading calendar and security-specific suspension/listing history, gap flags are `REVIEW`, not confirmed missing trading days.

In [5]:
PRICE_DIR_TOKENS = ("/prices/", "/prices_daily/")
price_paths = []
for p in discover_files():
    rel = "/" + str(p.relative_to(DATA_ROOT)).replace("\\\\","/").lower()
    if p.suffix.lower() == ".parquet" and any(t in rel for t in PRICE_DIR_TOKENS):
        if any(x in p.name.lower() for x in ["extraction_failed","extraction_progress"]):
            continue
        price_paths.append(p)

# Candidate column names across common vendor formats.
ALIASES = {
    "date": ["Date","DATE","PX_DATE","TRADE_DATE","timestamp"],
    "ticker": ["Ticker","ticker","Security","Security_ID","Symbol","Bloomberg_Ticker","ID"],
    "open": ["PX_OPEN","Open","OPEN","open_price"],
    "high": ["PX_HIGH","High","HIGH","high_price"],
    "low": ["PX_LOW","Low","LOW","low_price"],
    "close": ["PX_LAST","PX_CLOSE","Close","CLOSE","Adj Close","Last","LAST"],
    "volume": ["PX_VOLUME","Volume","VOLUME","volume"]
}

def standardize_price_frame(df, path):
    out = pd.DataFrame(index=df.index)
    found = {}
    for k, candidates in ALIASES.items():
        c = pick_col(df.columns, candidates)
        found[k] = c
        if c is not None:
            out[k] = df[c]
    if "date" not in out:
        raise ValueError(f"No recognizable date column. Columns={list(df.columns)}")
    out["date"] = to_datetime_series(out["date"])
    if "ticker" not in out:
        out["ticker"] = path.stem
    else:
        out["ticker"] = out["ticker"].astype("string").str.strip()
        out["ticker"] = out["ticker"].fillna(path.stem)
    for k in ["open","high","low","close","volume"]:
        if k in out:
            out[k] = numeric_series(out[k])
    out["source_file"] = str(path.relative_to(DATA_ROOT))
    return out, found

for p in price_paths:
    try:
        raw = read_table(p)
        std, found = standardize_price_frame(raw, p)
        price_frames.append(std)
        price_file_rows.append({
            "file": str(p.relative_to(DATA_ROOT)), "rows": len(raw),
            "ticker_values": std["ticker"].nunique(dropna=True),
            "min_date": std["date"].min(), "max_date": std["date"].max(),
            "columns_found": json.dumps(found), "readable": True
        })
    except Exception as e:
        price_file_rows.append({"file": str(p.relative_to(DATA_ROOT)), "readable": False, "error": repr(e)})
        add_issue("FAIL", "prices", "PRICE_FILE_READ_OR_SCHEMA_FAILURE", repr(e), p)

price_file_audit = pd.DataFrame(price_file_rows)
if price_frames:
    prices = pd.concat(price_frames, ignore_index=True)
else:
    prices = pd.DataFrame(columns=["date","ticker","open","high","low","close","volume","source_file"])

print("Price files:", len(price_paths), "| readable files:", int(price_file_audit.get("readable", pd.Series(dtype=bool)).sum()))
print("Rows:", len(prices), "| securities:", prices["ticker"].nunique() if len(prices) else 0)
print("Date range:", prices["date"].min(), "to", prices["date"].max())
display(price_file_audit.head(20))

Price files: 860 | readable files: 860
Rows: 1798828 | securities: 858
Date range: 2016-08-29 00:00:00 to 2026-08-27 00:00:00


,file,rows,ticker_values,min_date,max_date,columns_found,readable
0,DE/data/raw/prices/FSOL.parquet,2478,1,2016-08-29,2026-08-27,"{""date"": ""Date"", ""ticker"": ""Ticker"", ""open"": ""...",True
1,DE/data/raw/prices/TCIEXP.parquet,2405,1,2016-12-15,2026-08-27,"{""date"": ""Date"", ""ticker"": ""Ticker"", ""open"": ""...",True
2,DE/data/raw/prices/QUESS.parquet,2478,1,2016-08-29,2026-08-27,"{""date"": ""Date"", ""ticker"": ""Ticker"", ""open"": ""...",True
3,DE/data/raw/prices/PIRPHARM.parquet,957,1,2022-10-19,2026-08-27,"{""date"": ""Date"", ""ticker"": ""Ticker"", ""open"": ""...",True
4,DE/data/raw/prices/ALPM.parquet,2478,1,2016-08-29,2026-08-27,"{""date"": ""Date"", ""ticker"": ""Ticker"", ""open"": ""...",True
5,DE/data/raw/prices/PGHL.parquet,2478,1,2016-08-29,2026-08-27,"{""date"": ""Date"", ""ticker"": ""Ticker"", ""open"": ""...",True
6,DE/data/raw/prices/FIRSTCRY.parquet,508,1,2024-08-12,2026-08-27,"{""date"": ""Date"", ""ticker"": ""Ticker"", ""open"": ""...",True
7,DE/data/raw/prices/VRLL.parquet,2478,1,2016-08-29,2026-08-27,"{""date"": ""Date"", ""ticker"": ""Ticker"", ""open"": ""...",True
8,DE/data/raw/prices/NUVAMA.parquet,725,1,2023-09-26,2026-08-27,"{""date"": ""Date"", ""ticker"": ""Ticker"", ""open"": ""...",True
9,DE/data/raw/prices/CAST.parquet,1100,1,2016-08-29,2021-07-20,"{""date"": ""Date"", ""ticker"": ""Ticker"", ""open"": ""...",True


In [6]:
price_missing_rows = summarize_missing(prices, "prices")
price_issues = []

required_price_cols = ["date","ticker","close"]
for c in required_price_cols:
    if c not in prices.columns:
        add_issue("FAIL", "prices", "REQUIRED_COLUMN_MISSING", f"Required price field missing: {c}")

if len(prices):
    bad_date = prices["date"].isna()
    bad_ticker = prices["ticker"].isna() | prices["ticker"].astype(str).str.strip().eq("")
    for idx in prices.index[bad_date].tolist()[:10000]:
        add_issue("FAIL","prices","INVALID_DATE","Unparseable or missing price date",
                  prices.loc[idx,"source_file"], prices.loc[idx,"ticker"])
    for idx in prices.index[bad_ticker].tolist()[:10000]:
        add_issue("FAIL","prices","INVALID_TICKER","Missing/blank ticker",prices.loc[idx,"source_file"])

    valid_key = prices.loc[~bad_date & ~bad_ticker].copy()
    dup_mask = valid_key.duplicated(["ticker","date"], keep=False)
    if dup_mask.any():
        dups = valid_key.loc[dup_mask].sort_values(["ticker","date"])
        for _, r in dups.head(10000).iterrows():
            add_issue("FAIL","prices","DUPLICATE_TICKER_DATE",
                      "Duplicate security/date observation",r["source_file"],r["ticker"],r["date"])
        display(dups.head(30))

    # Infinity is distinct from NaN.
    for c in ["open","high","low","close","volume"]:
        if c in prices:
            infmask = np.isinf(prices[c].to_numpy(dtype=float, na_value=np.nan))
            if infmask.any():
                for idx in prices.index[infmask][:10000]:
                    add_issue("FAIL","prices","INFINITE_NUMERIC_VALUE",f"Infinite value in {c}",
                              prices.loc[idx,"source_file"],prices.loc[idx,"ticker"],prices.loc[idx,"date"])

    for c in ["open","high","low","close"]:
        if c in prices:
            mask = prices[c].notna() & (prices[c] <= 0)
            for idx in prices.index[mask][:10000]:
                add_issue("FAIL","prices","NONPOSITIVE_PRICE",f"{c} <= 0",
                          prices.loc[idx,"source_file"],prices.loc[idx,"ticker"],prices.loc[idx,"date"],prices.loc[idx,c])

    if "volume" in prices:
        mask = prices["volume"].notna() & (prices["volume"] < 0)
        for idx in prices.index[mask][:10000]:
            add_issue("FAIL","prices","NEGATIVE_VOLUME","Volume < 0",
                      prices.loc[idx,"source_file"],prices.loc[idx,"ticker"],prices.loc[idx,"date"],prices.loc[idx,"volume"])

    # OHLC ordering: only evaluate rows where all relevant values exist.
    if all(c in prices for c in ["open","high","low","close"]):
        complete = prices[["open","high","low","close"]].notna().all(axis=1)
        tol = MAX_OHLC_REL_TOLERANCE
        bad = complete & (
            (prices["high"] + tol*prices["high"].abs() < prices[["open","close","low"]].max(axis=1)) |
            (prices["low"] - tol*prices["low"].abs() > prices[["open","close","high"]].min(axis=1)) |
            (prices["high"] < prices["low"])
        )
        for idx in prices.index[bad][:10000]:
            r=prices.loc[idx]
            add_issue("FAIL","prices","OHLC_INCONSISTENCY","OHLC ordering violated",
                      r["source_file"],r["ticker"],r["date"],
                      {k:r[k] for k in ["open","high","low","close"]})

    # Missingness: distinguish close missing from partial OHLC.
    for c in ["open","high","low","close","volume"]:
        if c in prices:
            n=int(prices[c].isna().sum())
            if n:
                add_issue("REVIEW","prices","MISSING_PRICE_FIELD",f"{n} missing values in {c}")
    if "close" in prices:
        partial = prices[["open","high","low","volume"]].isna().any(axis=1) & prices["close"].notna()
        if partial.any():
            add_issue("REVIEW","prices","PARTIAL_OHLCV_ROWS",
                      f"{int(partial.sum())} rows have close but one or more other OHLCV fields missing")

    # Per-security chronology and calendar-gap review.
    gap_rows=[]
    returns_rows=[]
    for ticker, g in valid_key.sort_values("date").groupby("ticker", sort=False):
        g=g.sort_values("date")
        dates=g["date"].dropna()
        if len(dates)>1:
            delta=dates.diff().dt.days
            for i in delta[delta > MAX_GAP_CALENDAR_DAYS].index:
                r=g.loc[i]
                gap_rows.append({"ticker":ticker,"date":r["date"],"calendar_gap_days":int(delta.loc[i]),
                                 "source_file":r["source_file"]})
        if "close" in g:
            c=g["close"]
            ret=c.pct_change(fill_method=None)
            extreme=ret.abs()>EXTREME_ABS_DAILY_RETURN
            for i in g.index[extreme.fillna(False)]:
                r=g.loc[i]
                returns_rows.append({"ticker":ticker,"date":r["date"],"close":r["close"],
                                     "daily_return":ret.loc[i],"source_file":r["source_file"]})
    gap_df=pd.DataFrame(gap_rows)
    extreme_return_df=pd.DataFrame(returns_rows)
    if len(gap_df):
        add_issue("REVIEW","prices","LARGE_CALENDAR_GAPS",
                  f"{len(gap_df)} inter-observation gaps exceed {MAX_GAP_CALENDAR_DAYS} calendar days; requires exchange-calendar and lifecycle review")
    if len(extreme_return_df):
        add_issue("REVIEW","prices","EXTREME_DAILY_RETURNS",
                  f"{len(extreme_return_df)} close-to-close returns exceed {EXTREME_ABS_DAILY_RETURN:.0%}; check corporate actions, bad ticks and genuine moves")
    display(gap_df.head(30))
    display(extreme_return_df.head(30))
else:
    gap_df=pd.DataFrame()
    extreme_return_df=pd.DataFrame()
    add_issue("BLOCKER","prices","NO_PRICE_DATA","No price observations could be loaded.")

display(pd.DataFrame(price_missing_rows))

,date,ticker,open,high,low,close,volume,source_file
667654,2025-08-28,RELIANCE IN Equity,1387.05,1392.80,1366.5,1385.9,13584329.0,DE/data/raw/prices/RELIANCE.parquet
1798332,2025-08-28,RELIANCE IN Equity,1387.05,1392.80,1366.5,1385.9,13584329.0,DE/data/raw/prices_daily/TEST_RELIANCE_2025_20...
1798580,2025-08-28,RELIANCE IN Equity,1387.05,1392.80,1366.5,1385.9,13584329.0,DE/data/raw/prices_daily/RELIANCE_IN_2025_2026...
667655,2025-08-29,RELIANCE IN Equity,1384.20,1403.50,1350.0,1357.2,19629294.0,DE/data/raw/prices/RELIANCE.parquet
1798333,2025-08-29,RELIANCE IN Equity,1384.20,1403.50,1350.0,1357.2,19629294.0,DE/data/raw/prices_daily/TEST_RELIANCE_2025_20...
1798581,2025-08-29,RELIANCE IN Equity,1384.20,1403.50,1350.0,1357.2,19629294.0,DE/data/raw/prices_daily/RELIANCE_IN_2025_2026...
667656,2025-09-01,RELIANCE IN Equity,1357.85,1363.20,1340.6,1353.9,11546377.0,DE/data/raw/prices/RELIANCE.parquet
1798334,2025-09-01,RELIANCE IN Equity,1357.85,1363.20,1340.6,1353.9,11546377.0,DE/data/raw/prices_daily/TEST_RELIANCE_2025_20...
1798582,2025-09-01,RELIANCE IN Equity,1357.85,1363.20,1340.6,1353.9,11546377.0,DE/data/raw/prices_daily/RELIANCE_IN_2025_2026...
667657,2025-09-02,RELIANCE IN Equity,1354.80,1384.65,1354.5,1366.5,11983427.0,DE/data/raw/prices/RELIANCE.parquet


,ticker,date,calendar_gap_days,source_file
0,HDIL IN Equity,2023-05-02,8,DE/data/raw/prices/HDIL.parquet
1,HDIL IN Equity,2023-06-19,10,DE/data/raw/prices/HDIL.parquet
2,HDIL IN Equity,2023-10-03,8,DE/data/raw/prices/HDIL.parquet
3,HDIL IN Equity,2023-11-28,8,DE/data/raw/prices/HDIL.parquet
4,HDIL IN Equity,2023-12-26,8,DE/data/raw/prices/HDIL.parquet
5,HDIL IN Equity,2024-01-23,8,DE/data/raw/prices/HDIL.parquet
6,HDIL IN Equity,2024-02-19,10,DE/data/raw/prices/HDIL.parquet
7,HDIL IN Equity,2024-03-26,8,DE/data/raw/prices/HDIL.parquet
8,HDIL IN Equity,2024-09-02,10,DE/data/raw/prices/HDIL.parquet
9,HDIL IN Equity,2025-04-28,11,DE/data/raw/prices/HDIL.parquet


,ticker,date,close,daily_return,source_file
0,AGIG IN Equity,2019-08-19,61.75,-0.737066,DE/data/raw/prices/AGIG.parquet
1,CCD IN Equity,2021-04-26,46.10,0.749526,DE/data/raw/prices/CCD.parquet
2,SWANDEF IN Equity,2025-01-20,37.78,15.426087,DE/data/raw/prices/SWANDEF.parquet
3,BCG IN Equity,2025-07-14,20.71,1.020488,DE/data/raw/prices/BCG.parquet
4,RCOM IN Equity,2018-05-17,16.40,0.561905,DE/data/raw/prices/RCOM.parquet
5,ARVND IN Equity,2018-11-28,108.60,-0.650860,DE/data/raw/prices/ARVND.parquet
6,PATANJAL IN Equity,2020-01-24,1.11,-0.990060,DE/data/raw/prices/PATANJAL.parquet
7,PATANJAL IN Equity,2020-01-27,5.63,4.072072,DE/data/raw/prices/PATANJAL.parquet
8,JETIN IN Equity,2019-06-20,62.85,0.898792,DE/data/raw/prices/JETIN.parquet
9,TTCH IN Equity,2020-03-04,314.95,-0.565256,DE/data/raw/prices/TTCH.parquet


,dataset,column,rows,missing,missing_pct
0,prices,open,1798828,162,0.0090
1,prices,high,1798828,162,0.0090
2,prices,low,1798828,162,0.0090
3,prices,volume,1798828,672,0.0374


## 5. Security mapping and security-master validation

The existing mapping audit and `security_master` are treated as source artifacts. This section inspects their columns, tests duplicate and null identifiers, and tries to identify likely source/target identifier columns by conservative names.

It does **not** equate tickers merely by stripping suffixes. Such normalization is useful for generating candidate matches, but can incorrectly merge different securities or erase historical identity changes. Ambiguous mappings must be resolved with exchange/vendor identifiers and effective dates.

In [7]:
def load_first(paths, label):
    if not paths:
        print(f"No {label} file found.")
        return None, None
    # Prefer parquet for structured master, CSV for mapping audit where available.
    p=paths[0]
    try:
        d=read_table(p)
        print(f"{label}: {p.relative_to(DATA_ROOT)} | {d.shape}")
        display(d.head(10))
        return d,p
    except Exception as e:
        add_issue("FAIL","mapping","MAPPING_FILE_READ_FAILURE",repr(e),p)
        return None,p

security_master, security_master_path = load_first(security_master_files, "Security master")
mapping_audit, mapping_audit_path = load_first(mapping_audit_files, "Historical mapping audit")

def inspect_mapping_table(df, label, path=None):
    if df is None:
        return
    print(label, "columns:", list(df.columns))
    # Common column candidates. Keep detected fields visible in report.
    id_col=pick_col(df.columns,["security_id","internal_security_id","bbg_id","permanent_id","isin"])
    member_col=pick_col(df.columns,["membership_ticker","constituent_ticker","member_ticker","nifty_ticker","symbol"])
    price_col=pick_col(df.columns,["price_ticker","bloomberg_ticker","vendor_ticker","price_symbol","ticker"])
    status_col=pick_col(df.columns,["mapping_status","status","match_status"])
    count_col=pick_col(df.columns,["price_match_count","match_count","number_of_matches"])
    start_col=pick_col(df.columns,["valid_from","effective_from","start_date","from_date"])
    end_col=pick_col(df.columns,["valid_to","effective_to","end_date","to_date"])
    print("Detected:", {"security_id":id_col,"membership_id":member_col,"price_id":price_col,
                        "status":status_col,"match_count":count_col,"valid_from":start_col,"valid_to":end_col})
    if id_col and df[id_col].isna().any():
        add_issue("FAIL","mapping","NULL_INTERNAL_SECURITY_ID",f"{label}: null {id_col}",path)
    for col, name in [(member_col,"membership"),(price_col,"price")]:
        if col:
            n=int(df[col].isna().sum())
            if n:
                add_issue("REVIEW","mapping","NULL_MAPPING_IDENTIFIER",f"{label}: {n} null {name} identifiers in {col}",path)
    if member_col and price_col:
        dup=df.duplicated([member_col,price_col],keep=False)
        if dup.any():
            add_issue("REVIEW","mapping","DUPLICATE_MAPPING_PAIRS",
                      f"{label}: {int(dup.sum())} rows participate in duplicate membership/price pairs",path)
        # Count one-to-many and many-to-one relationships.
        m2p=df.groupby(member_col)[price_col].nunique(dropna=True)
        p2m=df.groupby(price_col)[member_col].nunique(dropna=True)
        one_many=m2p[m2p>1]
        many_one=p2m[p2m>1]
        if len(one_many):
            add_issue("REVIEW","mapping","ONE_TO_MANY_MEMBERSHIP_MAPPING",
                      f"{label}: {len(one_many)} membership identifiers map to multiple price identifiers",path,
                      evidence=one_many.head(50).to_dict())
        if len(many_one):
            add_issue("REVIEW","mapping","MANY_TO_ONE_PRICE_MAPPING",
                      f"{label}: {len(many_one)} price identifiers map to multiple membership identifiers",path,
                      evidence=many_one.head(50).to_dict())
        display(pd.DataFrame({
            "check":["unique membership IDs","unique price IDs","one-to-many membership IDs","many-to-one price IDs"],
            "count":[df[member_col].nunique(dropna=True),df[price_col].nunique(dropna=True),len(one_many),len(many_one)]
        }))
    if status_col:
        display(df[status_col].astype("string").value_counts(dropna=False).rename_axis("mapping_status").to_frame("rows"))
    if count_col:
        counts=numeric_series(df[count_col])
        display(counts.value_counts(dropna=False).sort_index().rename_axis("price_match_count").to_frame("rows"))
        if (counts.fillna(0)==0).any():
            add_issue("REVIEW","mapping","ZERO_PRICE_MATCHES",
                      f"{label}: {(counts.fillna(0)==0).sum()} rows report zero price matches",path)
        if (counts>1).any():
            add_issue("REVIEW","mapping","MULTIPLE_PRICE_MATCHES",
                      f"{label}: {(counts>1).sum()} rows report multiple price matches",path)
    if start_col and end_col:
        starts=to_datetime_series(df[start_col]); ends=to_datetime_series(df[end_col])
        bad=starts.notna() & ends.notna() & (starts>ends)
        if bad.any():
            add_issue("FAIL","mapping","INVALID_MAPPING_VALIDITY_INTERVAL",
                      f"{label}: {int(bad.sum())} intervals have valid_from after valid_to",path)

inspect_mapping_table(security_master,"security_master",security_master_path)
inspect_mapping_table(mapping_audit,"mapping_audit",mapping_audit_path)

Security master: DE/data/raw/constituents/security_master.parquet | (500, 4)


,Bloomberg_Index_Code,Bloomberg_ID,Index,Snapshot_Date
0,360ONE IS,360ONE IN Equity,NSE500 Index,2026-08-27
1,3M IS,3M IN Equity,NSE500 Index,2026-08-27
2,AADHARHF IS,AADHARHF IN Equity,NSE500 Index,2026-08-27
3,AAVAS IS,AAVAS IN Equity,NSE500 Index,2026-08-27
4,ABB IS,ABB IN Equity,NSE500 Index,2026-08-27
5,ABCAP IS,ABCAP IN Equity,NSE500 Index,2026-08-27
6,ABDL IS,ABDL IN Equity,NSE500 Index,2026-08-27
7,ABFRL IS,ABFRL IN Equity,NSE500 Index,2026-08-27
8,ABLBL IS,ABLBL IN Equity,NSE500 Index,2026-08-27
9,ABREL IS,ABREL IN Equity,NSE500 Index,2026-08-27


Historical mapping audit: DE/data/raw/constituents/historical_membership_price_mapping_audit.csv | (9857, 5)


,Date,Ticker,Membership_Root,Price_Match_Count,Mapping_Status
0,2016-09-01,1958850D IS,1958850D,1,Unique_Price_Match
1,2016-09-01,3M IS,3M,1,Unique_Price_Match
2,2016-09-01,63MOONS IS,63MOONS,1,Unique_Price_Match
3,2016-09-01,ABAN IS,ABAN,1,Unique_Price_Match
4,2016-09-01,ABB IS,ABB,1,Unique_Price_Match
5,2016-09-01,ABFRL IS,ABFRL,1,Unique_Price_Match
6,2016-09-01,ABGS IS,ABGS,1,Unique_Price_Match
7,2016-09-01,ABREL IS,ABREL,1,Unique_Price_Match
8,2016-09-01,ACC IS,ACC,1,Unique_Price_Match
9,2016-09-01,ACEM IS,ACEM,1,Unique_Price_Match


security_master columns: ['Bloomberg_Index_Code', 'Bloomberg_ID', 'Index', 'Snapshot_Date']
Detected: {'security_id': None, 'membership_id': None, 'price_id': None, 'status': None, 'match_count': None, 'valid_from': None, 'valid_to': None}
mapping_audit columns: ['Date', 'Ticker', 'Membership_Root', 'Price_Match_Count', 'Mapping_Status']
Detected: {'security_id': None, 'membership_id': None, 'price_id': 'Ticker', 'status': 'Mapping_Status', 'match_count': 'Price_Match_Count', 'valid_from': None, 'valid_to': None}


,rows
mapping_status,
Unique_Price_Match,9856
No_Price_Match,1


,rows
price_match_count,
0,1
1,9856


In [8]:
# Compare mapping identifiers to actual price identifiers, if a clear mapping column exists.
actual_price_ids=set(prices["ticker"].dropna().astype(str).str.strip().unique()) if len(prices) else set()
mapping_coverage_rows=[]

for label, df, path in [("security_master",security_master,security_master_path),
                        ("mapping_audit",mapping_audit,mapping_audit_path)]:
    if df is None: continue
    pc=pick_col(df.columns,["price_ticker","bloomberg_ticker","vendor_ticker","price_symbol","ticker"])
    mc=pick_col(df.columns,["membership_ticker","constituent_ticker","member_ticker","nifty_ticker","symbol"])
    if pc:
        vals=set(df[pc].dropna().astype(str).str.strip())
        matched=vals & actual_price_ids
        missing=vals-matched
        mapping_coverage_rows.append({"table":label,"identifier_column":pc,
                                      "unique_ids":len(vals),"exact_price_file_matches":len(matched),
                                      "not_found_in_loaded_price_ids":len(missing),
                                      "match_pct":100*len(matched)/len(vals) if vals else np.nan})
        if missing:
            add_issue("REVIEW","mapping","MAPPED_PRICE_ID_NOT_FOUND",
                      f"{label}: {len(missing)} mapped price identifiers not found exactly among loaded price IDs",
                      path,evidence=list(sorted(missing))[:100])
    if mc:
        mapping_coverage_rows.append({"table":label,"identifier_column":mc,
                                      "unique_ids":df[mc].nunique(dropna=True),
                                      "exact_price_file_matches":np.nan,
                                      "not_found_in_loaded_price_ids":np.nan,
                                      "match_pct":np.nan})

mapping_coverage_df=pd.DataFrame(mapping_coverage_rows)
display(mapping_coverage_df)
print("Loaded price identifiers sample:", sorted(actual_price_ids)[:30])

,table,identifier_column,unique_ids,exact_price_file_matches,not_found_in_loaded_price_ids,match_pct
0,mapping_audit,Ticker,861,0,861,0.0


Loaded price identifiers sample: ['1958850D IN Equity', '360ONE IN Equity', '3M IN Equity', '63MOONS IN Equity', 'AACL IN Equity', 'AADHARHF IN Equity', 'AAVAS IN Equity', 'ABAN IN Equity', 'ABB IN Equity', 'ABCAP IN Equity', 'ABFRL IN Equity', 'ABGS IN Equity', 'ABLBL IN Equity', 'ABNL IN Equity', 'ABREL IN Equity', 'ABSLAMC IN Equity', 'ACC IN Equity', 'ACCE IN Equity', 'ACEM IN Equity', 'ACI IN Equity', 'ACLGATI IN Equity', 'ACMESOLA IN Equity', 'ADANI IN Equity', 'ADANIENS IN Equity', 'ADANIGR IN Equity', 'ADE IN Equity', 'ADSEZ IN Equity', 'ADVENTHT IN Equity', 'ADVENZY IN Equity', 'AEGISLOG IN Equity']


## 6. Historical membership validation

Membership may be represented as dated snapshots (`date + security`) or as effective intervals (`valid_from + valid_to + security`). The notebook detects either format.

Snapshot counts are reported per date. An outlier is a review item, not an automatic failure: the index may not have exactly 500 constituents in every source snapshot, depending on source methodology and timing.

If the file only contains 20 snapshots, the notebook cannot infer exact constituent-change dates between snapshots. It will flag that limitation.

In [9]:
membership, membership_path = load_first(membership_files, "Historical membership")
membership_audit_rows=[]

if membership is not None:
    sec_col=pick_col(membership.columns,["security_id","internal_security_id","ticker","symbol","constituent","member","isin"])
    date_col=pick_col(membership.columns,["date","snapshot_date","as_of_date","effective_date"])
    start_col=pick_col(membership.columns,["valid_from","effective_from","start_date","from_date"])
    end_col=pick_col(membership.columns,["valid_to","effective_to","end_date","to_date"])
    print("Detected membership columns:", {"security":sec_col,"snapshot_date":date_col,"valid_from":start_col,"valid_to":end_col})
    if sec_col is None:
        add_issue("BLOCKER","membership","NO_SECURITY_KEY",
                  "Could not identify a security identifier column in membership data.",membership_path)
    if date_col:
        membership["_audit_date"]=to_datetime_series(membership[date_col])
        bad=membership["_audit_date"].isna()
        if bad.any():
            add_issue("FAIL","membership","INVALID_MEMBERSHIP_DATES",
                      f"{int(bad.sum())} membership rows have unparseable dates",membership_path)
        if sec_col:
            keydup=membership.duplicated(["_audit_date",sec_col],keep=False)
            if keydup.any():
                add_issue("FAIL","membership","DUPLICATE_MEMBERSHIP_SNAPSHOT_KEYS",
                          f"{int(keydup.sum())} rows duplicate date/security keys",membership_path)
            snapshots=(membership.dropna(subset=["_audit_date"])
                       .groupby("_audit_date")[sec_col].nunique().rename("unique_constituents").reset_index())
            display(snapshots)
            if len(snapshots)<30:
                add_issue("UNVERIFIED","membership","SPARSE_MEMBERSHIP_SNAPSHOTS",
                          f"Only {len(snapshots)} membership snapshots found. Exact daily membership changes cannot be reconstructed from sparse snapshots alone.",
                          membership_path,evidence=snapshots.to_dict("records"))
    elif start_col and end_col:
        membership["_audit_start"]=to_datetime_series(membership[start_col])
        membership["_audit_end"]=to_datetime_series(membership[end_col])
        bad=membership["_audit_start"].isna() | membership["_audit_end"].isna() | (membership["_audit_start"]>membership["_audit_end"])
        if bad.any():
            add_issue("FAIL","membership","INVALID_MEMBERSHIP_INTERVALS",
                      f"{int(bad.sum())} rows have invalid effective intervals",membership_path)
        if sec_col:
            dup=membership.duplicated([sec_col,"_audit_start","_audit_end"],keep=False)
            if dup.any():
                add_issue("REVIEW","membership","DUPLICATE_MEMBERSHIP_INTERVALS",
                          f"{int(dup.sum())} rows duplicate security/effective interval keys",membership_path)
    else:
        add_issue("BLOCKER","membership","NO_MEMBERSHIP_TIME_FIELD",
                  "Could not identify snapshot date or effective membership interval columns.",membership_path)
else:
    add_issue("BLOCKER","membership","MEMBERSHIP_FILE_MISSING","No historical membership file found.")

Historical membership: DE/data/raw/constituents/nifty500_membership_validation.csv | (20, 4)


,Date,Unique_Tickers,Expected_500,Difference_From_500
0,2016-09-01,501,500,1
1,2017-03-01,501,500,1
2,2017-09-01,501,500,1
3,2018-03-01,501,500,1
4,2018-09-01,338,500,-162
5,2019-03-01,501,500,1
6,2019-09-01,501,500,1
7,2020-03-01,501,500,1
8,2020-09-01,501,500,1
9,2021-03-01,501,500,1


Detected membership columns: {'security': None, 'snapshot_date': 'Date', 'valid_from': None, 'valid_to': None}


## 7. Fundamentals and point-in-time validity

The notebook checks schema, duplicate company/period records, missing numeric values, non-finite values, date parsing and availability/publication fields.

A fiscal period end is not the date the market learned the information. If no announcement/publication/availability timestamp exists, the fundamentals are not certified for point-in-time backtesting. Do not fill the publication date using the fiscal period end.

In [10]:
fundamental_audits=[]
fundamental_frames=[]

for p in fundamental_files:
    try:
        d=read_table(p)
        fundamental_frames.append((p,d))
        sec=pick_col(d.columns,["security_id","ticker","symbol","isin","company_id","bbg_ticker"])
        period=pick_col(d.columns,["period_end","fiscal_period_end","date","as_of_date","report_date","quarter_end"])
        pub=pick_col(d.columns,["announcement_date","announced_at","publication_date","published_at","available_date","available_at","filing_date"])
        revision=pick_col(d.columns,["revision_date","revised_at","updated_at","restatement_date"])
        print("\\n",p.relative_to(DATA_ROOT),d.shape)
        print({"security":sec,"period":period,"publication_or_availability":pub,"revision":revision})
        display(d.head(5))
        if sec is None or period is None:
            add_issue("BLOCKER","fundamentals","FUNDAMENTALS_KEY_UNDETECTED",
                      f"Could not detect security and period fields in {p.name}",p)
            continue
        d["_audit_security"]=d[sec].astype("string").str.strip()
        d["_audit_period"]=to_datetime_series(d[period])
        if d["_audit_period"].isna().any():
            add_issue("FAIL","fundamentals","INVALID_FUNDAMENTAL_PERIOD_DATES",
                      f"{int(d['_audit_period'].isna().sum())} invalid period dates",p)
        dup=d.duplicated(["_audit_security","_audit_period"],keep=False)
        if dup.any():
            add_issue("REVIEW","fundamentals","DUPLICATE_FUNDAMENTAL_PERIODS",
                      f"{int(dup.sum())} rows duplicate security/period; determine whether these are versions, restatements or duplicate extraction.",p)
        if pub:
            d["_audit_available"]=to_datetime_series(d[pub])
            missing=int(d["_audit_available"].isna().sum())
            if missing:
                add_issue("BLOCKER","fundamentals","MISSING_PUBLICATION_TIMESTAMPS",
                          f"{missing} rows lack a usable publication/availability timestamp.",p)
            future=(d["_audit_available"].notna() & d["_audit_period"].notna() &
                    (d["_audit_available"]<d["_audit_period"]))
            # Not always impossible (some data can be released before period end), so flag for review.
            if future.any():
                add_issue("REVIEW","fundamentals","PUBLICATION_BEFORE_PERIOD_END",
                          f"{int(future.sum())} records have availability earlier than period end; verify semantics.",p)
        else:
            add_issue("BLOCKER","fundamentals","NO_PUBLICATION_TIMESTAMP_FIELD",
                      "No announcement/publication/availability field detected. Period-end dates alone do not establish point-in-time availability.",p)
        numeric_cols=[c for c in d.columns if pd.api.types.is_numeric_dtype(d[c])]
        for c in numeric_cols:
            inf=np.isinf(d[c].to_numpy(dtype=float,na_value=np.nan))
            if inf.any():
                add_issue("FAIL","fundamentals","INFINITE_FUNDAMENTAL_VALUE",
                          f"{int(inf.sum())} infinite values in {c}",p)
        for c in d.columns:
            if pd.api.types.is_numeric_dtype(d[c]):
                n=int(d[c].isna().sum())
                if n:
                    fundamental_audits.append({"file":str(p.relative_to(DATA_ROOT)),"column":c,
                                               "rows":len(d),"missing":n,"missing_pct":100*n/len(d)})
    except Exception as e:
        add_issue("FAIL","fundamentals","FUNDAMENTALS_READ_FAILURE",repr(e),p)

fundamental_missing_df=pd.DataFrame(fundamental_audits)
display(fundamental_missing_df.sort_values(["missing_pct"],ascending=False) if len(fundamental_missing_df) else fundamental_missing_df)

\n DE/data/raw/fundamentals/nifty500_historical_fundamental_incremental_field_exceptions.parquet (0, 0)
{'security': None, 'period': None, 'publication_or_availability': None, 'revision': None}


""


\n DE/data/raw/fundamentals/potentially_invalid_fundamental_values.csv (511, 4)
{'security': None, 'period': 'Date', 'publication_or_availability': None, 'revision': None}


,Bloomberg_Security,Date,Value,Field
0,1958850D IN Equity,2020-03-31,-423.3402,BOOK_VAL_PER_SH
1,1958850D IN Equity,2020-09-30,-433.4034,BOOK_VAL_PER_SH
2,ABAN IN Equity,2020-03-31,-2624.7554,BOOK_VAL_PER_SH
3,ABAN IN Equity,2020-09-30,-2621.9683,BOOK_VAL_PER_SH
4,ABAN IN Equity,2021-03-31,-2882.0889,BOOK_VAL_PER_SH


\n DE/data/raw/fundamentals/nifty500_historical_fundamentals_failures.parquet (17, 2)
{'security': None, 'period': None, 'publication_or_availability': None, 'revision': None}


,Bloomberg_Security,Reason
0,AEGISVOP IN Equity,No historical fieldData returned
1,ANTHEM IN Equity,No historical fieldData returned
2,ATHERENE IN Equity,No historical fieldData returned
3,BELRISE IN Equity,No historical fieldData returned
4,CPPLUS IN Equity,No historical fieldData returned


... (39 outputs left out)

,Bloomberg_Security,Fundamental_First_Date,Price_First_Date,Years_From_Price_To_Fundamentals,Classification
0,1958850D IN Equity,2020-03-31,2016-08-29,3.586585,Fundamentals start much later
1,360ONE IN Equity,2020-03-31,2019-09-19,0.531143,Fundamentals start much later
2,3M IN Equity,2020-03-31,2016-08-29,3.586585,Fundamentals start much later
3,63MOONS IN Equity,2020-03-31,2016-08-29,3.586585,Fundamentals start much later
4,AACL IN Equity,2020-03-31,2016-08-29,3.586585,Fundamentals start much later


\n DE/data/raw/fundamentals/fundamentals_missingness_by_quarter.csv (20, 28)
{'security': None, 'period': None, 'publication_or_availability': None, 'revision': None}


,Quarter,Securities,SALES_REV_TURN_Missing,SALES_REV_TURN_Missing_Pct,NET_INCOME_Missing,NET_INCOME_Missing_Pct,EBITDA_Missing,EBITDA_Missing_Pct,EBIT_Missing,EBIT_Missing_Pct,BOOK_VAL_PER_SH_Missing,BOOK_VAL_PER_SH_Missing_Pct,IS_EPS_Missing,IS_EPS_Missing_Pct,CUR_MKT_CAP_Missing,CUR_MKT_CAP_Missing_Pct,EQY_SH_OUT_Missing,EQY_SH_OUT_Missing_Pct,PE_RATIO_Missing,PE_RATIO_Missing_Pct,PX_TO_BOOK_RATIO_Missing,PX_TO_BOOK_RATIO_Missing_Pct,PX_TO_SALES_RATIO_Missing,PX_TO_SALES_RATIO_Missing_Pct,DVD_SH_12M_Missing,DVD_SH_12M_Missing_Pct,TOT_RETURN_INDEX_GROSS_DVDS_Missing,TOT_RETURN_INDEX_GROSS_DVDS_Missing_Pct
0,2020-03-31,730,31,4.246575,31,4.246575,69,9.452055,68,9.315068,49,6.712329,32,4.383562,10,1.369863,10,1.369863,164,22.465753,72,9.863014,68,9.315068,192,26.301370,16,2.191781
1,2020-06-30,737,35,4.748982,34,4.613297,72,9.769335,72,9.769335,655,88.873813,36,4.884668,18,2.442334,18,2.442334,192,26.051560,79,10.719132,73,9.905020,201,27.272727,23,3.120760
2,2020-09-30,739,31,4.194858,31,4.194858,70,9.472260,70,9.472260,61,8.254398,32,4.330176,30,4.059540,20,2.706360,224,30.311231,84,11.366712,74,10.013532,204,27.604871,31,4.194858
3,2020-12-31,742,22,2.964960,20,2.695418,62,8.355795,62,8.355795,655,88.274933,23,3.099730,27,3.638814,15,2.021563,221,29.784367,83,11.185984,64,8.625337,263,35.444744,28,3.773585
4,2021-03-31,756,27,3.571429,27,3.571429,70,9.259259,68,8.994709,48,6.349206,28,3.703704,31,4.100529,20,2.645503,193,25.529101,83,10.978836,73,9.656085,292,38.624339,32,4.232804


""


## 8. Corporate actions and price adjustment status

The archive is searched for actual files whose names suggest splits, bonuses, dividends, rights, mergers, demergers or corporate actions. The existence of a filename is not proof that the data is complete.

The notebook also flags large price discontinuities as review candidates. A discontinuity alone does not prove a split or bad price: it may be a genuine market move, a corporate action, a bad tick, or a security-identifier transition.

For a trustworthy total-return backtest, establish whether the vendor's close series is raw, split-adjusted, or dividend-adjusted, and document the adjustment methodology.

In [11]:
print("Corporate action files discovered:")
for p in corporate_action_files:
    try:
        d=read_table(p)
        print(p.relative_to(DATA_ROOT),d.shape,list(d.columns))
        display(d.head(10))
    except Exception as e:
        add_issue("FAIL","corporate_actions","CORPORATE_ACTION_FILE_READ_FAILURE",repr(e),p)

if not corporate_action_files:
    print("No corporate-action source file discovered in the extracted data.")

# Large moves from price audit are candidates only.
if len(extreme_return_df):
    display(extreme_return_df.sort_values("daily_return",key=lambda s:s.abs(),ascending=False).head(100))

add_issue("UNVERIFIED","corporate_actions","PRICE_ADJUSTMENT_BASIS_UNVERIFIED",
          "The archive audit cannot prove whether price history is raw, split-adjusted or dividend-adjusted without vendor documentation or authoritative corporate-action reconciliation.")

Corporate action files discovered:
No corporate-action source file discovered in the extracted data.


,ticker,date,close,daily_return,source_file
2,SWANDEF IN Equity,2025-01-20,37.78,15.426087,DE/data/raw/prices/SWANDEF.parquet
10,KSI IN Equity,2025-03-10,9.71,7.024793,DE/data/raw/prices/KSI.parquet
12,ALOK IN Equity,2020-02-19,18.05,4.469697,DE/data/raw/prices/ALOK.parquet
7,PATANJAL IN Equity,2020-01-27,5.63,4.072072,DE/data/raw/prices/PATANJAL.parquet
41,TATATECH IN Equity,2023-11-30,1313.00,1.626000,DE/data/raw/prices/TATATECH.parquet
36,LATENTVI IN Equity,2021-11-23,488.75,1.480964,DE/data/raw/prices/LATENTVI.parquet
45,BAJAJHFL IN Equity,2024-09-16,165.00,1.357143,DE/data/raw/prices/BAJAJHFL.parquet
14,GVKP IN Equity,2022-08-12,5.70,1.280000,DE/data/raw/prices/GVKP.parquet
23,IRCTC IN Equity,2019-10-14,145.55,1.274219,DE/data/raw/prices/IRCTC.parquet
29,RBA IN Equity,2020-12-14,135.00,1.250000,DE/data/raw/prices/RBA.parquet


## 9. Feature dataset audit and independent recomputation

This section checks all discovered feature datasets for duplicate keys, missing values, infinities, date ranges and identifier coverage. If common columns exist, it independently computes close-based returns, simple moving averages, rolling volatility and rolling highs/lows from the loaded raw price series and compares overlapping observations.

The calculations are deliberately conservative:
- They use the close column as supplied.
- They do not silently fill missing prices.
- They use rolling observations, not calendar-day interpolation.
- The feature dataset must use compatible definitions/lookback conventions for a direct comparison.

RSI/MACD/ROC can be added once the source definitions (EMA seed, smoothing convention, adjusted/raw close basis) are known. A numeric mismatch does not automatically prove look-ahead; first align definitions.

In [12]:
feature_frames=[]
feature_missing_rows=[]

for p in feature_files:
    try:
        d=read_table(p)
        feature_frames.append((p,d))
        print("\\nFeature file:",p.relative_to(DATA_ROOT),d.shape)
        print(list(d.columns))
        display(d.head(3))
        datec=pick_col(d.columns,["date","trade_date"])
        tickc=pick_col(d.columns,["ticker","symbol","security_id","bbg_ticker"])
        if datec and tickc:
            dd=to_datetime_series(d[datec])
            dup=pd.DataFrame({"date":dd,"ticker":d[tickc].astype("string")}).duplicated(["date","ticker"],keep=False)
            if dup.any():
                add_issue("FAIL","features","DUPLICATE_FEATURE_KEYS",
                          f"{int(dup.sum())} rows duplicate date/ticker keys",p)
            if dd.isna().any():
                add_issue("FAIL","features","INVALID_FEATURE_DATES",
                          f"{int(dd.isna().sum())} invalid feature dates",p)
        else:
            add_issue("REVIEW","features","FEATURE_KEYS_UNDETECTED",
                      "Could not identify date and ticker columns; feature-level key tests skipped.",p)
        for c in d.columns:
            if pd.api.types.is_numeric_dtype(d[c]):
                miss=int(d[c].isna().sum())
                inf=int(np.isinf(d[c].to_numpy(dtype=float,na_value=np.nan)).sum())
                if miss or inf:
                    feature_missing_rows.append({"file":str(p.relative_to(DATA_ROOT)),"column":c,
                                                 "rows":len(d),"missing":miss,"infinite":inf})
                if inf:
                    add_issue("FAIL","features","INFINITE_FEATURE_VALUES",
                              f"{inf} infinite values in {c}",p)
    except Exception as e:
        add_issue("FAIL","features","FEATURE_FILE_READ_FAILURE",repr(e),p)

feature_missing_df=pd.DataFrame(feature_missing_rows)
display(feature_missing_df)

\nFeature file: audit_output/feature_audit.csv (35, 6)
['feature', 'missing', 'missing_pct', 'unique_values', 'min', 'max']


,feature,missing,missing_pct,unique_values,min,max
0,PX_OPEN,162,0.009,182810,0.09,162294.9
1,PX_HIGH,162,0.009,190973,0.09,163600.0
2,PX_LOW,162,0.009,192342,0.09,160215.0


\nFeature file: DE/data/processed/nifty500_historical_stock_features.parquet (1798332, 37)
['Date', 'Ticker', 'PX_OPEN', 'PX_HIGH', 'PX_LOW', 'PX_LAST', 'PX_VOLUME', 'Return_1M', 'Return_3M', 'Return_6M', 'Return_12M', 'DMA_20', 'Price_20DMA', 'DMA_50', 'Price_50DMA', 'DMA_100', 'Price_100DMA', 'DMA_200', 'Price_200DMA', '52W_High', '52W_Low', 'Pct_Below_52W_High', 'Pct_Above_52W_Low', 'RSI', 'ROC', 'MACD', 'Daily_Return', 'Volatility_20D', 'Volatility_60D', 'Volatility_120D', 'Volatility_252D', 'Current_Drawdown', 'Max_Drawdown_3M', 'Max_Drawdown_12M', 'Volume_Change', 'Volume_20D_Avg', 'Volume_to_20D_Avg']


,Date,Ticker,PX_OPEN,PX_HIGH,PX_LOW,PX_LAST,PX_VOLUME,Return_1M,Return_3M,Return_6M,Return_12M,DMA_20,Price_20DMA,DMA_50,Price_50DMA,DMA_100,Price_100DMA,DMA_200,Price_200DMA,52W_High,52W_Low,Pct_Below_52W_High,Pct_Above_52W_Low,RSI,ROC,MACD,Daily_Return,Volatility_20D,Volatility_60D,Volatility_120D,Volatility_252D,Current_Drawdown,Max_Drawdown_3M,Max_Drawdown_12M,Volume_Change,Volume_20D_Avg,Volume_to_20D_Avg
0,2016-08-29,1958850D IN Equity,44.90,44.9,43.45,43.8,1293320.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
1,2016-08-30,1958850D IN Equity,43.95,45.9,43.55,44.9,2670138.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.087749,0.025114,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,1.064561,NaN,NaN
2,2016-08-31,1958850D IN Equity,45.30,45.7,44.30,44.7,1537626.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.139544,-0.004454,NaN,NaN,NaN,NaN,-0.004454,NaN,NaN,-0.424140,NaN,NaN


,file,column,rows,missing,infinite
0,DE/data/processed/nifty500_historical_stock_fe...,PX_OPEN,1798332,162,0
1,DE/data/processed/nifty500_historical_stock_fe...,PX_HIGH,1798332,162,0
2,DE/data/processed/nifty500_historical_stock_fe...,PX_LOW,1798332,162,0
3,DE/data/processed/nifty500_historical_stock_fe...,PX_VOLUME,1798332,672,0
4,DE/data/processed/nifty500_historical_stock_fe...,Return_1M,1798332,18018,0
5,DE/data/processed/nifty500_historical_stock_fe...,Return_3M,1798332,53972,0
6,DE/data/processed/nifty500_historical_stock_fe...,Return_6M,1798332,107900,0
7,DE/data/processed/nifty500_historical_stock_fe...,Return_12M,1798332,215150,0
8,DE/data/processed/nifty500_historical_stock_fe...,DMA_20,1798332,16302,0
9,DE/data/processed/nifty500_historical_stock_fe...,Price_20DMA,1798332,16302,0


In [13]:
# Recompute a transparent set of common indicators from raw close prices.
recomputed = None
if len(prices) and {"date","ticker","close"}.issubset(prices.columns):
    base=prices[["date","ticker","close"]].dropna(subset=["date","ticker"]).copy()
    base=base.sort_values(["ticker","date"])
    # Do not resolve duplicate date/ticker records silently.
    base=base.drop_duplicates(["ticker","date"],keep=False)
    base["close"]=numeric_series(base["close"])
    base=base.dropna(subset=["close"])
    g=base.groupby("ticker",group_keys=False)
    base["calc_daily_return"]=g["close"].pct_change(fill_method=None)
    for w in [20,50,100,200,252]:
        base[f"calc_sma_{w}"]=g["close"].transform(lambda s:s.rolling(w,min_periods=w).mean())
    for w in [20,60,120,252]:
        base[f"calc_volatility_{w}"]=g["calc_daily_return"].transform(
            lambda s:s.rolling(w,min_periods=w).std(ddof=1)*np.sqrt(252))
    base["calc_52w_high"]=g["close"].transform(lambda s:s.rolling(252,min_periods=252).max())
    base["calc_52w_low"]=g["close"].transform(lambda s:s.rolling(252,min_periods=252).min())
    recomputed=base
    print("Recomputed feature rows:",len(recomputed))
else:
    print("Raw prices unavailable; independent feature recomputation skipped.")

# Map common feature column names to recomputed series.
FEATURE_ALIASES = {
    "daily_return":["Daily_Return","DailyReturn"],
    "sma20":["DMA_20","Price_20DMA","SMA_20","MA_20"],
    "sma50":["DMA_50","Price_50DMA","SMA_50","MA_50"],
    "sma100":["DMA_100","Price_100DMA","SMA_100","MA_100"],
    "sma200":["DMA_200","Price_200DMA","SMA_200","MA_200"],
    "sma252":["SMA_252","DMA_252","Price_252DMA"],
    "high252":["52W_High","52WHigh","52_Week_High"],
    "low252":["52W_Low","52WLow","52_Week_Low"],
    "vol20":["Volatility_20D","Volatility20D"],
    "vol60":["Volatility_60D","Volatility60D"],
    "vol120":["Volatility_120D","Volatility120D"],
    "vol252":["Volatility_252D","Volatility252D"]
}
RECOMP_MAP={"daily_return":"calc_daily_return","sma20":"calc_sma_20","sma50":"calc_sma_50",
            "sma100":"calc_sma_100","sma200":"calc_sma_200","sma252":"calc_sma_252",
            "high252":"calc_52w_high","low252":"calc_52w_low","vol20":"calc_volatility_20",
            "vol60":"calc_volatility_60","vol120":"calc_volatility_120","vol252":"calc_volatility_252"}

comparison_rows=[]
if recomputed is not None:
    for p,d in feature_frames:
        dc=pick_col(d.columns,["date","trade_date"]); tc=pick_col(d.columns,["ticker","symbol","security_id","bbg_ticker"])
        if not dc or not tc: continue
        f=d.copy()
        f["_date"]=to_datetime_series(f[dc]); f["_ticker"]=f[tc].astype("string").str.strip()
        merged=f.merge(recomputed,left_on=["_date","_ticker"],right_on=["date","ticker"],how="inner",validate="many_to_one")
        for logical,aliases in FEATURE_ALIASES.items():
            fc=pick_col(f.columns,aliases)
            rc=RECOMP_MAP[logical]
            if fc and rc in merged:
                a=numeric_series(merged[fc]); b=numeric_series(merged[rc])
                valid=a.notna() & b.notna() & np.isfinite(a) & np.isfinite(b)
                if valid.any():
                    diff=(a[valid]-b[valid]).abs()
                    tol=FEATURE_ABS_TOLERANCE + FEATURE_REL_TOLERANCE*b[valid].abs()
                    bad=diff>tol
                    comparison_rows.append({
                        "file":str(p.relative_to(DATA_ROOT)),"feature":fc,"overlap_rows":int(valid.sum()),
                        "mismatch_rows":int(bad.sum()),"mismatch_pct":100*bad.mean(),
                        "mae":float(diff.mean()),"max_abs_error":float(diff.max())
                    })
                    if bad.any():
                        add_issue("REVIEW","features","FEATURE_RECOMPUTATION_MISMATCH",
                                  f"{fc}: {int(bad.sum())}/{int(valid.sum())} overlapping rows differ from independent calculation.",p,
                                  evidence={"mismatch_pct":100*bad.mean(),"mae":float(diff.mean())})
comparison_df=pd.DataFrame(comparison_rows)
display(comparison_df)

Recomputed feature rows: 1798084


,file,feature,overlap_rows,mismatch_rows,mismatch_pct,mae,max_abs_error
0,DE/data/processed/nifty500_historical_stock_fe...,Daily_Return,1797226,0,0.000000,0.000000,0.000000
1,DE/data/processed/nifty500_historical_stock_fe...,DMA_20,1781782,0,0.000000,0.000000,0.000000
2,DE/data/processed/nifty500_historical_stock_fe...,DMA_50,1756096,0,0.000000,0.000000,0.000000
3,DE/data/processed/nifty500_historical_stock_fe...,DMA_100,1713296,0,0.000000,0.000000,0.000000
4,DE/data/processed/nifty500_historical_stock_fe...,DMA_200,1627942,0,0.000000,0.000000,0.000000
5,DE/data/processed/nifty500_historical_stock_fe...,52W_High,1583783,0,0.000000,0.000000,0.000000
6,DE/data/processed/nifty500_historical_stock_fe...,52W_Low,1583783,0,0.000000,0.000000,0.000000
7,DE/data/processed/nifty500_historical_stock_fe...,Volatility_20D,1780924,1780922,99.999888,0.355864,51.304536
8,DE/data/processed/nifty500_historical_stock_fe...,Volatility_60D,1746680,1746680,100.000000,0.370697,29.626794
9,DE/data/processed/nifty500_historical_stock_fe...,Volatility_120D,1695320,1695320,100.000000,0.377389,20.955724


## 10. Cross-dataset coverage and consistency

This compares the identifier/date domains across datasets only where their semantics permit it. Membership snapshots are not expected to have the same daily keys as price history. Instead, the audit reports security coverage and date-range overlap.

The notebook does not infer membership intervals from sparse snapshots. It does not equate company names or ticker roots without an explicit mapping.

In [14]:
coverage_rows=[]
if len(prices):
    coverage_rows.append({"dataset":"prices","rows":len(prices),"securities":prices["ticker"].nunique(),
                          "min_date":prices["date"].min(),"max_date":prices["date"].max()})
if membership is not None:
    sc=pick_col(membership.columns,["security_id","internal_security_id","ticker","symbol","constituent","member","isin"])
    dc=pick_col(membership.columns,["date","snapshot_date","as_of_date","effective_date"])
    if sc:
        coverage_rows.append({"dataset":"membership","rows":len(membership),"securities":membership[sc].nunique(),
                              "min_date":to_datetime_series(membership[dc]).min() if dc else None,
                              "max_date":to_datetime_series(membership[dc]).max() if dc else None})
for p,d in fundamental_frames:
    sc=pick_col(d.columns,["security_id","ticker","symbol","isin","company_id","bbg_ticker"])
    dc=pick_col(d.columns,["period_end","fiscal_period_end","date","as_of_date","report_date","quarter_end"])
    coverage_rows.append({"dataset":f"fundamentals:{p.name}","rows":len(d),
                          "securities":d[sc].nunique() if sc else None,
                          "min_date":to_datetime_series(d[dc]).min() if dc else None,
                          "max_date":to_datetime_series(d[dc]).max() if dc else None})
coverage_df=pd.DataFrame(coverage_rows)
display(coverage_df)

# Raw exact-ID comparisons are diagnostic only; mapping is required for valid joins.
if membership is not None and len(prices):
    sc=pick_col(membership.columns,["security_id","internal_security_id","ticker","symbol","constituent","member","isin"])
    if sc:
        member_ids=set(membership[sc].dropna().astype(str).str.strip())
        price_ids=set(prices["ticker"].dropna().astype(str).str.strip())
        exact=member_ids & price_ids
        print("Exact raw identifier overlap (diagnostic only):",len(exact),
              "/",len(member_ids),"membership IDs")
        print("Do not interpret low exact overlap as missing prices until mapping is applied.")

,dataset,rows,securities,min_date,max_date
0,prices,1798828,858.0,2016-08-29,2026-08-27
1,fundamentals:nifty500_historical_fundamental_i...,0,NaN,NaT,NaT
2,fundamentals:potentially_invalid_fundamental_v...,511,NaN,2020-03-31,2024-12-31
3,fundamentals:nifty500_historical_fundamentals_...,17,NaN,NaT,NaT
4,fundamentals:fundamentals_field_missingness.csv,13,NaN,NaT,NaT
5,fundamentals:clean_quarterly_price_fundamental...,20,NaN,NaT,NaT
6,fundamentals:nifty500_historical_fundamentals_...,21,NaN,NaT,NaT
7,fundamentals:nifty500_historical_fundamentals_...,7,NaN,NaT,NaT
8,fundamentals:fundamentals_coverage_report.csv,9,NaN,NaT,NaT
9,fundamentals:fundamentals_numeric_validity.csv,13,NaN,NaT,NaT


## 11. Extraction logs and failure reports

Only source files explicitly found in the raw price folder are inspected here. The audit output directory is outside `DATA_ROOT`, so generated reports cannot recursively audit themselves.

In [15]:
extraction_logs=[p for p in discover_files()
                 if p.parent.name.lower() in {"prices","price","logs","reports"}
                 and any(t in p.name.lower() for t in ["failed","failure","progress","error"])
                 and p.suffix.lower() in {".csv",".parquet"}]
for p in extraction_logs:
    try:
        d=read_table(p)
        print(p.relative_to(DATA_ROOT),d.shape,list(d.columns))
        display(d.head(20))
        if "failed" in p.name.lower() or "error" in p.name.lower():
            if len(d):
                add_issue("REVIEW","extraction","SOURCE_EXTRACTION_FAILURE_LOG_NONEMPTY",
                          f"{p.name} contains {len(d)} rows; review every failed extraction.",p)
    except Exception as e:
        add_issue("FAIL","extraction","EXTRACTION_LOG_READ_FAILURE",repr(e),p)

DE/data/raw/prices/extraction_progress.csv (858, 3) ['security', 'status', 'rows']


,security,status,rows
0,1958850D IN Equity,success,539
1,3M IN Equity,success,2478
2,63MOONS IN Equity,success,2478
3,ABAN IN Equity,success,2428
4,ABB IN Equity,success,2478
5,ABFRL IN Equity,success,2478
6,ABGS IN Equity,success,677
7,ABREL IN Equity,success,2478
8,ACC IN Equity,success,2478
9,ACEM IN Equity,success,2478


DE/data/raw/prices/extraction_failed.parquet (3, 2) ['security', 'reason']


,security,reason
0,360ONE IN,no_data
1,RAYMONDL IN,no_data
2,SANOFICO IN,no_data


DE/data/raw/prices/extraction_failed.csv (3, 2) ['security', 'reason']


,security,reason
0,360ONE IN,no_data
1,RAYMONDL IN,no_data
2,SANOFICO IN,no_data


## 12. Readiness assessment

Readiness is strategy-dependent. This notebook produces separate assessments for:
- **Price-only technical strategy**: requires validated prices, security identity, historical universe, and corporate-action treatment.
- **Fundamental strategy**: additionally requires point-in-time fundamentals.
- **Any historical Nifty 500 strategy**: requires reliable historical membership effective dates.

A blocker is not removed just because the code could not detect a problem. Missing evidence is itself a blocker where the backtest depends on it.

In [16]:
issues_df=pd.DataFrame(issues)
if len(issues_df):
    severity_order={"BLOCKER":0,"FAIL":1,"REVIEW":2,"UNVERIFIED":3,"PASS":4}
    issues_df["_order"]=issues_df["severity"].map(severity_order).fillna(9)
    issues_df=issues_df.sort_values(["_order","area","code"]).drop(columns="_order")

def has_issue(area=None, severities=("BLOCKER","FAIL"), codes=None):
    d=issues_df
    if d.empty: return False
    if area: d=d[d["area"]==area]
    d=d[d["severity"].isin(severities)]
    if codes: d=d[d["code"].isin(codes)]
    return len(d)>0

def issue_count(severity):
    return int((issues_df["severity"]==severity).sum()) if len(issues_df) else 0

# Evidence gaps that cannot be proven by local structural checks.
price_ok = len(prices)>0 and not has_issue("prices",("BLOCKER","FAIL"))
mapping_evidence = (security_master is not None) or (mapping_audit is not None)
mapping_blocked = not mapping_evidence or has_issue("mapping",("BLOCKER","FAIL"))
membership_blocked = membership is None or has_issue("membership",("BLOCKER","FAIL"))
membership_sparse = (len(membership_files)>0 and len(issues_df) and
                     (issues_df["code"]=="SPARSE_MEMBERSHIP_SNAPSHOTS").any())
corp_action_verified = bool(corporate_action_files)  # existence only; not correctness
fundamentals_pti = False
if fundamental_frames:
    # Conservative: every fundamentals source must expose a recognized publication/availability field.
    fundamentals_pti = all(
        pick_col(d.columns,["announcement_date","announced_at","publication_date","published_at","available_date","available_at","filing_date"])
        is not None for _,d in fundamental_frames
    )
    if fundamentals_pti and has_issue("fundamentals",("BLOCKER","FAIL")):
        fundamentals_pti=False

readiness_rows=[
    {"strategy":"Price-only technical","status":"BLOCKED" if (not price_ok or mapping_blocked or membership_blocked or membership_sparse or not corp_action_verified) else "CONDITIONAL",
     "reason":"Requires prices, reliable mapping, effective historical membership and verified corporate-action basis."},
    {"strategy":"Fundamental + technical","status":"BLOCKED" if (not price_ok or mapping_blocked or membership_blocked or membership_sparse or not corp_action_verified or not fundamentals_pti) else "CONDITIONAL",
     "reason":"Also requires point-in-time publication/availability timestamps and revision handling."}
]
readiness_df=pd.DataFrame(readiness_rows)
display(readiness_df)
display(issues_df.groupby(["severity","area"]).size().unstack(fill_value=0) if len(issues_df) else pd.DataFrame())

,strategy,status,reason
0,Price-only technical,BLOCKED,"Requires prices, reliable mapping, effective h..."
1,Fundamental + technical,BLOCKED,Also requires point-in-time publication/availa...


area,corporate_actions,extraction,features,fundamentals,mapping,membership,prices
severity,,,,,,,
BLOCKER,0,0,0,24,0,1,0
FAIL,0,0,0,0,0,0,744
REVIEW,0,2,5,0,2,0,7
UNVERIFIED,2,0,0,0,0,0,0


## 13. Export all audit outputs

Outputs include:
- `file_inventory.csv`
- `parquet_schema_audit.csv`
- `parquet_read_errors.csv`
- `price_file_audit.csv`
- `price_calendar_gaps.csv`
- `extreme_price_returns.csv`
- `mapping_coverage.csv`
- `fundamentals_missingness.csv`
- `feature_missingness.csv`
- `feature_recomputation_comparison.csv`
- `cross_dataset_coverage.csv`
- `issues.csv`
- `readiness.csv`
- `audit_summary.json`

These are reports only. This notebook does not delete, fill, adjust, rename, or overwrite source observations.

In [17]:
# Refresh issue dataframe after all checks.
issues_df=pd.DataFrame(issues)
if len(issues_df):
    severity_order={"BLOCKER":0,"FAIL":1,"REVIEW":2,"UNVERIFIED":3,"PASS":4}
    issues_df["_order"]=issues_df["severity"].map(severity_order).fillna(9)
    issues_df=issues_df.sort_values(["_order","area","code"]).drop(columns="_order")

outputs = {
    "file_inventory.csv": inventory_df,
    "parquet_schema_audit.csv": schema_df,
    "parquet_read_errors.csv": read_errors_df,
    "price_file_audit.csv": price_file_audit,
    "price_missingness.csv": pd.DataFrame(price_missing_rows),
    "price_calendar_gaps.csv": gap_df if "gap_df" in globals() else pd.DataFrame(),
    "extreme_price_returns.csv": extreme_return_df if "extreme_return_df" in globals() else pd.DataFrame(),
    "mapping_coverage.csv": mapping_coverage_df if "mapping_coverage_df" in globals() else pd.DataFrame(),
    "fundamentals_missingness.csv": fundamental_missing_df if "fundamental_missing_df" in globals() else pd.DataFrame(),
    "feature_missingness.csv": feature_missing_df if "feature_missing_df" in globals() else pd.DataFrame(),
    "feature_recomputation_comparison.csv": comparison_df if "comparison_df" in globals() else pd.DataFrame(),
    "cross_dataset_coverage.csv": coverage_df if "coverage_df" in globals() else pd.DataFrame(),
    "issues.csv": issues_df,
    "readiness.csv": readiness_df
}
for name, df in outputs.items():
    df.to_csv(REPORT_ROOT/name,index=False)

summary = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "data_root": str(DATA_ROOT.resolve()),
    "report_root": str(REPORT_ROOT.resolve()),
    "files_discovered_excluding_metadata": len(all_files),
    "parquet_files": sum(p.suffix.lower()==".parquet" for p in all_files),
    "csv_files": sum(p.suffix.lower()==".csv" for p in all_files),
    "parquet_read_failures": len(read_errors_df),
    "price_files": len(price_paths),
    "price_rows_loaded": int(len(prices)),
    "price_securities_loaded": int(prices["ticker"].nunique()) if len(prices) else 0,
    "membership_files": [str(p.relative_to(DATA_ROOT)) for p in membership_files],
    "mapping_audit_files": [str(p.relative_to(DATA_ROOT)) for p in mapping_audit_files],
    "security_master_files": [str(p.relative_to(DATA_ROOT)) for p in security_master_files],
    "fundamentals_files": [str(p.relative_to(DATA_ROOT)) for p in fundamental_files],
    "feature_files": [str(p.relative_to(DATA_ROOT)) for p in feature_files],
    "corporate_action_files": [str(p.relative_to(DATA_ROOT)) for p in corporate_action_files],
    "issue_counts": {s:issue_count(s) for s in ["BLOCKER","FAIL","REVIEW","UNVERIFIED"]},
    "readiness": readiness_df.to_dict("records"),
    "limitations": [
        "No authoritative exchange trading calendar was supplied; calendar gaps are review flags, not confirmed missing sessions.",
        "No vendor documentation was supplied to prove price adjustment basis.",
        "Corporate action file presence does not prove completeness or correctness.",
        "Sparse membership snapshots cannot establish exact constituent effective dates between snapshots.",
        "Fundamentals are not point-in-time valid without reliable public-availability timestamps and revision history.",
        "Feature comparisons depend on matching indicator definitions, lookback and initialization conventions."
    ]
}
with open(REPORT_ROOT/"audit_summary.json","w",encoding="utf-8") as f:
    json.dump(summary,f,indent=2,default=str)

print("Audit complete.")
print(json.dumps(summary,indent=2,default=str))
print("\\nReports saved to:",REPORT_ROOT)

Audit complete.
{
  "generated_utc": "2026-11-06T12:59:53.749013+00:00",
  "data_root": "/home/rishi/Documents/vscode/BacktestEngine/dataextracted",
  "report_root": "/home/rishi/Documents/vscode/BacktestEngine/audit_v2_output",
  "files_discovered_excluding_metadata": 931,
  "parquet_files": 877,
  "csv_files": 36,
  "parquet_read_failures": 0,
  "price_files": 860,
  "price_rows_loaded": 1798828,
  "price_securities_loaded": 858,
  "membership_files": [
    "DE/data/raw/constituents/nifty500_membership_validation.csv",
    "DE/data/raw/constituents/nifty500_historical_membership.parquet"
  ],
  "mapping_audit_files": [
    "DE/data/raw/constituents/historical_membership_price_mapping_audit.csv"
  ],
  "security_master_files": [
    "DE/data/raw/constituents/security_master.parquet"
  ],
  "fundamentals_files": [
    "DE/data/raw/fundamentals/nifty500_historical_fundamental_incremental_field_exceptions.parquet",
    "DE/data/raw/fundamentals/potentially_invalid_fundamental_values.csv"

## 14. Interpretation rules — do not skip

- **FAIL** means a defined structural or numerical test was violated. Investigate the source row; do not auto-delete it.
- **REVIEW** means an anomaly or ambiguity needs domain-level investigation.
- **UNVERIFIED** means the archive lacks evidence required to prove a claim.
- **BLOCKER** means the relevant backtest cannot be certified until the missing evidence or failure is resolved.
- **CONDITIONAL** means the automated checks did not find a hard blocker in the selected gate. It is not a guarantee of correctness.

### What this notebook cannot establish by itself

1. Whether every historical price is the true exchange/vendor value.
2. Whether the price history is split-adjusted or dividend-adjusted correctly.
3. Exact historical Nifty 500 constituent change dates from sparse snapshots.
4. Public availability of fundamentals without announcement/filing timestamps and revisions.
5. Whether a large price move is a genuine market move or an unrecorded corporate action.
6. Whether an indicator's vendor-specific implementation matches the independently calculated formula unless its exact definition is known.

Those require source documentation, exchange/vendor reference data, corporate-action records, or independent trusted datasets. The notebook intentionally reports these as unresolved rather than manufacturing a clean bill of health.